# Overview

This notebook id dedicated for evaluating NER model on a benchmark dataset.

# Step 0 - Setup
Run the code below to mount your Google Drive and most of the necessary packages to carry out the evaluation

**Action:**
No code changes required. When prompted, connect your Google account

In [1]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [2]:
%%capture
!cd /content/
!rm -rf ./CASM_utils/
!git clone -b master https://github.com/ay94/multilingual-ner.git
!pip install -e CASM_utils/
!cd /content/CASM_utils


import CASM_utils
import importlib
from CASM_utils import utils, ner
importlib.reload(ner)

In [3]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

import os
import pandas as pd

Mounted at /content/drive/


In [4]:
# direct the file handler to the data folder
FOLDER = '/content/drive/MyDrive/CASM/FAST/German/NER/Benchmark'
fh = utils.FileHandler(FOLDER)

 # Read NER Dataset --> the output should be list of words corresponding to list of labels
 Read NER data class, it contains three different functionalities to read NER data.
  The NER data in the literature normally have consistent internal structure and flexible external structure.
  The internal structure is that it comes in word-label pair, this is consistent across all datasets.
  The external structure normally differ from dataset to another, which is divided to three main categories:
  - Data that comes in one text file, the read_ner_file function can be used in this case.
  - Data that comes in text files split into, train, val and test, this type you can either read individual file separately or put them all in one folder and read_ner_directory function.
  - Data that comes in directory where the directory contians various text files divide by topic (e.g, AQMAR), this type of data normally wikipedia articles that has been scraped and preprocessed into named entities structure.
  - Finally data available on huggingface and this can be loaded using load_dataset function and pass it to the read_dataset class method.
  
  Most of the datasets fall under one of these types if your data is different you can add function to this class dedicated to your data.

xtreme

In [5]:
xtreme_label_map = {
    'O': 0, 'B-PER': 1, 'I-PER': 2,
    'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6
}
xtreme = ner.ReadNERData()
xtreme_words, xtreme_labels = xtreme.read_dataset('xtreme', xtreme_label_map, lang='PAN-X.de')

/usr/local/lib/python3.10/dist-packages/datasets/load.py:1429: FutureWarning: The repository for xtreme contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/xtreme
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

germeval_14

In [6]:
germeval_14_label_map = {
    'O': 0,
    'B-LOC': 1, 'I-LOC': 2, 'B-LOCderiv': 3, 'I-LOCderiv': 4, 'B-LOCpart': 5, 'I-LOCpart': 6,
    'B-ORG': 7, 'I-ORG': 8, 'B-ORGderiv': 9, 'I-ORGderiv': 10, 'B-ORGpart': 11, 'I-ORGpart': 12,
    'B-OTH': 13, 'I-OTH': 14, 'B-OTHderiv': 15, 'I-OTHderiv': 16, 'B-OTHpart': 17, 'I-OTHpart': 18,
    'B-PER': 19, 'I-PER': 20, 'B-PERderiv': 21, 'I-PERderiv': 22, 'B-PERpart': 23, 'I-PERpart': 24
}



germeval_14 = ner.ReadNERData()
germeval_14_words, germeval_14_labels = germeval_14.read_dataset('germeval_14', germeval_14_label_map)

Generating test Split


  0%|          | 0/5100 [00:00<?, ?it/s]

### Check for dataset alignment
The first thing to do after loading the data is to check that it is aligned with the standard annotation scheme using check_labels function. NER datasets have various annotation schemes, the standard one we are interested in is the conll annotation scheme where the data should be divided into, *LOC*, *PERS*, *ORG*, *MISC* entities and each entity have BI boudary (e.g, B-LOC, I-LOC) and outside named entity O. This is the standard annotation scheme we are aiming for and some dataset comes with fine grained annotations or even different labels. This requires realigning the dataset labels to the standard scheme by defining a dataset label alignment dictionary and use the align_dataset function.

xtreme

In [7]:
ner.check_labels(xtreme_labels)

{'B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O'}

collection3

In [8]:
print(ner.check_labels(germeval_14_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC
germeval_14_label_alignment = {
            "B-LOCderiv": "B-LOC", "I-LOCderiv": "I-LOC",
            "B-LOCpart": "B-LOC", "I-LOCpart": "I-LOC",
            "B-ORGderiv": "B-ORG", "I-ORGderiv": "I-ORG",
            "B-ORGpart": "B-ORG", "I-ORGpart": "I-ORG",
            "B-PERderiv": "B-ORG", "I-PERderiv": "I-ORG",
            "B-PERpart": "B-ORG", "I-PERpart": "I-ORG",
            "B-OTHderiv": "O", "I-OTHderiv": "O",
            "B-OTHpart": "O", "I-OTHpart": "O",
            "B-OTH": "O", "I-OTH": "O",



}
# Align the dataset labels to the standard labels
germeval_14_labels = ner.align_dataset(germeval_14_labels, germeval_14_label_alignment)
print(ner.check_labels(germeval_14_labels))


{'I-PER', 'B-LOC', 'B-PERderiv', 'O', 'B-ORG', 'B-OTHderiv', 'B-LOCpart', 'I-ORG', 'I-LOCderiv', 'B-PER', 'I-LOC', 'I-OTH', 'B-PERpart', 'I-PERpart', 'B-LOCderiv', 'B-ORGderiv', 'B-OTHpart', 'I-ORGpart', 'B-ORGpart', 'B-OTH'}
{'B-LOC', 'I-ORG', 'B-PER', 'I-LOC', 'O', 'B-ORG', 'I-PER'}


# Model Evaluation
Model evaluation is dvided into three steps:
- Loading the model using get_model funtion
- Generating the evaluation benchmark using generate_evaluation_data function
- Apply the model to the benchmakr and compute the performance using eval_fn

All of these steps can be achieved by calling evaluate_model function. It is worth noting that all models comes with their own labeling scheme and some models have different annotation scheme from the standard one we are using, this requires using label alignment dictionary to align the model's output.

In [9]:

model_name = "gunghio/xlm-roberta-base-finetuned-panx-ner"
model_name_output = 'gunghio-xlm-xtreme'
model_evaluation = ner.ModelEvaluation(model_name)

In [10]:
fh.create_folder(f'outputs/{model_name_output}')

Folder 'outputs/gunghio-xlm-xtreme' created successfully.


In [11]:
model_evaluation.model.config.id2label

{0: 'O',
 1: 'B-PER',
 2: 'I-PER',
 3: 'B-ORG',
 4: 'I-ORG',
 5: 'B-LOC',
 6: 'I-LOC'}

#### xtreme

In [12]:
data_name = "xtreme"
xtreme_evaluation_output = model_evaluation.evaluate_model(xtreme_words, xtreme_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [13]:
ner.check_labels(xtreme_evaluation_output.get_output('Seqeval')['y_true'])

{'B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O'}

In [14]:
ner.check_labels(xtreme_evaluation_output.get_output('Seqeval')['y_pred'])

{'B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O'}

In [15]:
xtreme_seqeval = xtreme_evaluation_output.get_classification('Seqeval')
xtreme_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.8662,0.8909,0.8784,4961
1,ORG,0.8064,0.8114,0.8089,4157
2,PER,0.9333,0.9194,0.9263,4750
3,micro,0.8707,0.8768,0.8738,13868
4,macro,0.8686,0.8739,0.8712,13868
5,weighted,0.8712,0.8768,0.8740,13868


In [16]:
xtreme_sklearn = xtreme_evaluation_output.get_classification('Sklearn')
xtreme_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.8870,0.9063,0.8965,4961
1,B-ORG,0.8434,0.8369,0.8401,4157
2,B-PER,0.9508,0.9320,0.9413,4750
3,I-LOC,0.8378,0.8419,0.8398,2289
4,I-ORG,0.8957,0.8855,0.8906,6043
5,I-PER,0.9624,0.9532,0.9578,6792
6,O,0.9892,0.9912,0.9902,68654
7,accuracy,0.9647,97646,None,None
8,macro,0.9095,0.9067,0.9080,97646
9,weighted,0.9647,0.9647,0.9647,97646


In [17]:
xtreme_seqeval.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-seqeval.csv'),
    index=False
)
xtreme_sklearn.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-sklearn.csv'),
    index=False
)

#### germeval_14

In [18]:
data_name = "germeval_14"
germeval_14_evaluation_output = model_evaluation.evaluate_model(germeval_14_words, germeval_14_labels)

  0%|          | 0/319 [00:00<?, ?it/s]

In [19]:
germeval_14_seqeval = germeval_14_evaluation_output.get_classification('Seqeval')
germeval_14_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.6841,0.6263,0.6539,2376
1,ORG,0.2228,0.6383,0.3303,1385
2,PER,0.7553,0.7383,0.7467,1639
3,micro,0.4626,0.6633,0.5450,5400
4,macro,0.5541,0.6676,0.5770,5400
5,weighted,0.5874,0.6633,0.5991,5400


In [20]:
germeval_14_sklearn = germeval_14_evaluation_output.get_classification('Sklearn')
germeval_14_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.7121,0.6444,0.6765,2376
1,B-ORG,0.2530,0.6960,0.3711,1385
2,B-PER,0.7799,0.7480,0.7636,1639
3,I-LOC,0.4712,0.6124,0.5326,307
4,I-ORG,0.2397,0.8895,0.3776,706
5,I-PER,0.8005,0.9726,0.8782,912
6,O,0.9894,0.9413,0.9648,89174
7,accuracy,0.9261,96499,None,None
8,macro,0.6066,0.7863,0.6521,96499
9,weighted,0.9596,0.9261,0.9393,96499


In [21]:
germeval_14_seqeval.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-seqeval.csv'),
    index=False
)
germeval_14_sklearn.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-sklearn.csv'),
    index=False
)
